---
## 4.6 XGBoost (eXtra Gradient Boost)

### 개념 요약
XGBoost는 GBM의 단점(느린 학습 속도, 과적합 규제 부족)을 보완한 부스팅 알고리즘이다.

**주요 특징**
- 병렬 학습 지원으로 GBM 대비 빠른 학습 속도
- L1, L2 규제를 통한 과적합 방지
- 자체 교차검증 및 조기 중단(early stopping) 기능 내장
- 결측치 자동 처리

**주요 하이퍼파라미터**
- `max_depth` : 트리의 최대 깊이 (과적합 제어)
- `eta` (learning_rate) : 학습률, 일반적으로 0.01 ~ 0.2
- `objective` : 학습 목적 함수 (이진분류는 `binary:logistic`)
- `eval_metric` : 평가 지표 (logloss, error, auc 등)

### 손실 함수
$$
Obj = \sum_{i=1}^{n} L(y_i, \hat{y_i}) + \sum_{k=1}^{K} \Omega(f_k)
$$
- $L$ : 손실 함수 (예: logloss)
- $\Omega$ : 트리 복잡도에 대한 규제항

* XGBoost 버전 확인

In [ ]:
import xgboost

print(xgboost.__version__)

### 파이썬 Native XGBoost 적용 – 위스콘신 Breast Cancer 데이터 셋

In [ ]:
import xgboost as xgb
from xgboost import plot_importance
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# 위스콘신 유방암 데이터셋 로드 (이진 분류 문제)
dataset = load_breast_cancer()
X_features= dataset.data
y_label = dataset.target

# DataFrame으로 변환하여 데이터 구조 확인
cancer_df = pd.DataFrame(data=X_features, columns=dataset.feature_names)
cancer_df['target']= y_label
cancer_df.head(3)

In [ ]:
# 타겟 클래스 확인: 0 = malignant(악성), 1 = benign(양성)
print(dataset.target_names)
print(cancer_df['target'].value_counts())

In [ ]:
# 전체 데이터 중 80%는 학습용 데이터, 20%는 테스트용 데이터 추출
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label,
                                         test_size=0.2, random_state=156 )
print(X_train.shape , X_test.shape)

In [ ]:
# 파이썬 Native XGBoost는 DMatrix 자료구조를 입력으로 사용
dtrain = xgb.DMatrix(data=X_train , label=y_train)
dtest = xgb.DMatrix(data=X_test , label=y_test)

In [ ]:
# 하이퍼파라미터를 딕셔너리 형태로 정의
params = { 'max_depth':3,              # 트리 최대 깊이
           'eta': 0.1,                 # 학습률
           'objective':'binary:logistic',  # 이진분류, 시그모이드 확률 출력
           'eval_metric':'logloss',    # 평가 지표
           'early_stoppings':100
        }
num_rounds = 400  # 부스팅 라운드 수 (트리 개수)

In [ ]:
# train 데이터 셋은 'train', evaluation(test) 데이터 셋은 'eval'로 명기
wlist = [(dtrain,'train'),(dtest,'eval') ]
# 하이퍼 파라미터와 early stopping 파라미터를 train() 함수의 파라미터로 전달
xgb_model = xgb.train(params = params , dtrain=dtrain , num_boost_round=num_rounds , evals=wlist )

In [ ]:
# predict() 결과는 확률값으로 반환됨 (사이킷런의 predict_proba와 유사)
pred_probs = xgb_model.predict(dtest)
print('predict( ) 수행 결과값을 10개만 표시, 예측 확률 값으로 표시됨')
print(np.round(pred_probs[:10],3))

# 예측 확률이 0.5 보다 크면 1, 그렇지 않으면 0으로 예측값 결정하여 List 객체인 preds에 저장 
preds = [ 1 if x > 0.5 else 0 for x in pred_probs ]
print('예측값 10개만 표시:',preds[:10])

In [ ]:
# 분류 성능 평가 함수
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score

def get_clf_eval(y_test , pred):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    roc_auc = roc_auc_score(y_test, pred)
    print('오차 행렬')
    print(confusion)
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
# 파이썬 Native XGBoost 평가
get_clf_eval(y_test , preds)

In [ ]:
# 피처 중요도 시각화
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax = plt.subplots(figsize=(10, 12))
plot_importance(xgb_model, ax=ax)

### 사이킷런 Wrapper XGBoost 개요 및 적용

사이킷런 호환 인터페이스를 제공하는 `XGBClassifier` / `XGBRegressor` 사용.
- `fit()`, `predict()`, `predict_proba()` 등 사이킷런 API와 동일하게 사용 가능
- DMatrix 변환 없이 DataFrame / ndarray를 그대로 입력으로 받음
- 파라미터명도 사이킷런 스타일 (`eta` → `learning_rate`, `num_boost_round` → `n_estimators` 등)

In [ ]:
# 사이킷런 래퍼 XGBoost 클래스인 XGBClassifier 임포트
from xgboost import XGBClassifier

evals = [(X_test, y_test)]
xgb_wrapper = XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=3)
# 사이킷런 API와 동일하게 fit으로 학습, early stopping은 fit 인자로 전달
xgb_wrapper.fit(X_train , y_train,  early_stopping_rounds=400,eval_set=evals, eval_metric="logloss",  verbose=True)
w_preds = xgb_wrapper.predict(X_test)

In [ ]:
get_clf_eval(y_test , w_preds)

In [ ]:
# early_stopping_rounds를 100으로 설정 → 검증 지표가 100라운드 동안 개선 없으면 학습 중단
from xgboost import XGBClassifier

xgb_wrapper = XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=3)
evals = [(X_test, y_test)]
xgb_wrapper.fit(X_train, y_train, early_stopping_rounds=100, eval_metric="logloss", 
                eval_set=evals, verbose=True)
ws100_preds = xgb_wrapper.predict(X_test)

In [ ]:
get_clf_eval(y_test , ws100_preds)

In [ ]:
# early_stopping_rounds를 10으로 설정하고 재 학습
# → 너무 작게 설정하면 학습이 충분히 안 되어 성능이 떨어질 수 있음 (underfitting 위험)
xgb_wrapper.fit(X_train, y_train, early_stopping_rounds=10, 
                eval_metric="logloss", eval_set=evals,verbose=True)

ws10_preds = xgb_wrapper.predict(X_test)
get_clf_eval(y_test , ws10_preds)

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax = plt.subplots(figsize=(10, 12))
# 사이킷런 래퍼 클래스를 입력해도 무방
plot_importance(xgb_wrapper, ax=ax)

---
## 4.7 LightGBM

### 개념 요약
LightGBM은 Microsoft에서 개발한 그래디언트 부스팅 프레임워크로, **XGBoost보다 빠르고 메모리 효율적**이라는 장점이 있다.

### XGBoost vs LightGBM 트리 분할 방식
- **XGBoost (Level-wise)** : 균형 트리 분할 → 트리의 균형은 유지되지만 시간 소요가 큼
- **LightGBM (Leaf-wise)** : 최대 손실값(max delta loss)을 가지는 리프 노드를 계속 분할 → 비대칭 트리 생성, 학습 시간 단축

### 단점
- 데이터가 적을 경우(보통 1만 건 이하) 과적합 가능성이 높음
- 리프 중심 분할로 인해 깊은 트리가 만들어질 수 있어 `max_depth`, `num_leaves` 등으로 적절한 제어 필요

### 주요 하이퍼파라미터
- `n_estimators` : 부스팅 라운드 수
- `learning_rate` : 학습률
- `num_leaves` : 하나의 트리가 가질 수 있는 최대 리프 개수 (LightGBM 핵심 파라미터)
- `max_depth` : 트리 최대 깊이

In [ ]:
# LightGBM 버전 확인
import lightgbm

print(lightgbm.__version__)

### LightGBM 적용 – 위스콘신 Breast Cancer Prediction

동일한 데이터셋(`load_breast_cancer`)을 사용해 XGBoost와 비교한다.

In [ ]:
# LightGBM의 사이킷런 래퍼 LGBMClassifier 임포트
from lightgbm import LGBMClassifier

import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

dataset = load_breast_cancer()

cancer_df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)
cancer_df['target']= dataset.target
X_features = cancer_df.iloc[:, :-1]
y_label = cancer_df.iloc[:, -1]

# 전체 데이터 중 80%는 학습용, 20%는 테스트용
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label, test_size=0.2, random_state=156 )

# 학습 데이터를 다시 학습 90% / 검증 10%로 분리 (early stopping용 검증셋 확보)
X_tr, X_val, y_tr, y_val= train_test_split(X_train, y_train, test_size=0.1, random_state=156 )

# XGBoost와 동일하게 n_estimators는 400으로 설정
lgbm_wrapper = LGBMClassifier(n_estimators=400, learning_rate=0.05)

# LightGBM도 XGBoost와 동일하게 조기 중단 수행 가능
evals = [(X_tr, y_tr), (X_val, y_val)]
lgbm_wrapper.fit(X_tr, y_tr, early_stopping_rounds=50, eval_metric="logloss", eval_set=evals, verbose=True)
preds = lgbm_wrapper.predict(X_test)
pred_proba = lgbm_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
# 평가 함수 재정의 (AUC를 확률값 기반으로 계산하도록 pred_proba 인자 추가)
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score

def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    # ROC-AUC는 확률값 기반으로 계산해야 정확함
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
get_clf_eval(y_test, preds, pred_proba)

In [ ]:
# plot_importance()를 이용하여 feature 중요도 시각화
from lightgbm import plot_importance
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax = plt.subplots(figsize=(10, 12))
plot_importance(lgbm_wrapper, ax=ax)

---
## 4.8 베이지안 최적화 기반의 HyperOpt를 이용한 하이퍼 파라미터 튜닝

### 베이지안 최적화란?
기존 Grid Search / Random Search는 탐색 공간을 사전에 정해놓고 모든 조합(또는 무작위 조합)을 시도한다.  
**베이지안 최적화**는 **이전 시도 결과(사후 확률)**를 기반으로 다음 탐색 지점을 추론한다.

- **대체 모델(Surrogate Model)** : 목적 함수의 형태를 확률 모델(예: 가우시안 프로세스, TPE)로 근사
- **획득 함수(Acquisition Function)** : 다음에 어디를 탐색할지 결정 (탐색-활용 균형)

### HyperOpt 핵심 구성 요소
1. **Search Space** : `hp.quniform`, `hp.uniform`, `hp.choice` 등으로 탐색 공간 정의
2. **Objective Function** : 최소화할 목적 함수 (정확도처럼 클수록 좋은 값은 `-1`을 곱해서 반환)
3. **fmin()** : 최적값을 찾는 함수, `algo=tpe.suggest`로 TPE 알고리즘 사용
4. **Trials** : 시도 이력을 저장하는 객체

In [ ]:
#pip install hyperopt
import hyperopt

print(hyperopt.__version__)

### HyperOpt 사용법 – 간단한 예제

$f(x, y) = x^2 - 20y$ 의 최솟값을 찾는 문제로 HyperOpt 사용법을 익힌다.  
이론적 최솟값은 $x=0$, $y=15$ 부근에서 발생.

In [ ]:
from hyperopt import hp

# hp.quniform(label, low, high, q): low~high 범위에서 q 간격으로 균등 샘플링
# x: -10 ~ 10 (1 간격), y: -15 ~ 15 (1 간격)
search_space = {'x': hp.quniform('x', -10, 10, 1), 'y': hp.quniform('y', -15, 15, 1) }

In [ ]:
from hyperopt import STATUS_OK

# 목적 함수 생성. 변숫값과 변수 검색 공간을 가지는 딕셔너리를 인자로 받고, 특정 값을 반환
def objective_func(search_space):
    x = search_space['x']
    y = search_space['y']
    retval = x**2 - 20*y  # 최소화 대상 식
    
    return retval

In [ ]:
from hyperopt import fmin, tpe, Trials
import numpy as np

# Trials: 입력 결괏값을 저장한 객체
trial_val = Trials()

# fmin(): 목적 함수의 최솟값을 반환하는 최적 입력 변숫값을 5번의 입력값 시도(max_evals=5)로 찾아냄
# - algo=tpe.suggest : 베이지안 최적화 알고리즘 (TPE = Tree-structured Parzen Estimator)
# - rstate : 재현성을 위한 랜덤 시드
best_01 = fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=5
               , trials=trial_val, rstate=np.random.default_rng(seed=0))
print('best:', best_01)

In [ ]:
trial_val = Trials()

# max_evals를 20회로 늘려서 재테스트 → 더 좋은 해를 찾을 가능성 증가
best_02 = fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=20
               , trials=trial_val, rstate=np.random.default_rng(seed=0))
print('best:', best_02)

In [ ]:
# fmin()에 인자로 들어가는 Trials 객체의 result 속성에 파이썬 리스트로 목적 함수 반환값들이 저장됨
# 리스트 내부의 개별 원소는 {'loss':함수 반환값, 'status':반환 상태값} 와 같은 딕셔너리
print(trial_val.results)

In [ ]:
# Trials 객체의 vals 속성에 {'입력변수명':개별 수행 시마다 입력된 값 리스트} 형태로 저장됨
print(trial_val.vals)

In [ ]:
import pandas as pd

# results에서 loss 키값에 해당하는 밸류들을 추출하여 list로 생성
losses = [loss_dict['loss'] for loss_dict in trial_val.results]

# DataFrame으로 정리 (x, y, loss를 한눈에 비교)
result_df = pd.DataFrame({'x': trial_val.vals['x'], 'y': trial_val.vals['y'], 'losses': losses})
result_df

### HyperOpt를 이용한 XGBoost 하이퍼 파라미터 최적화

이제 본격적으로 위스콘신 유방암 데이터셋에 대해 XGBoost의 하이퍼파라미터를 HyperOpt로 튜닝한다.

**주의 사항**
- `hp.quniform`은 실수형을 반환하므로, 정수형 파라미터(`max_depth`, `min_child_weight`)는 `int()`로 변환 필요
- HyperOpt는 **최소화** 문제만 풀기 때문에 정확도(높을수록 좋음)에는 `-1`을 곱해서 반환

In [ ]:
# 데이터셋 재로딩 (셀 독립 실행 보장)
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

dataset = load_breast_cancer()

cancer_df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)
cancer_df['target']= dataset.target
X_features = cancer_df.iloc[:, :-1]
y_label = cancer_df.iloc[:, -1]

In [ ]:
# 전체 데이터 중 80%는 학습용, 20%는 테스트용
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label, test_size=0.2, random_state=156 )

# 학습 데이터를 다시 학습과 검증 데이터로 분리
X_tr, X_val, y_tr, y_val= train_test_split(X_train, y_train, test_size=0.1, random_state=156 )

In [ ]:
from hyperopt import hp

# XGBoost 주요 하이퍼파라미터 탐색 공간 정의
# - max_depth: 5 ~ 20 사이 정수 (1 간격)
# - min_child_weight: 1 ~ 2 사이 정수 (1 간격)
# - learning_rate: 0.01 ~ 0.2 사이 균등 분포 실수
# - colsample_bytree: 0.5 ~ 1 사이 균등 분포 실수 (트리당 사용할 피처 비율)
xgb_search_space = {'max_depth': hp.quniform('max_depth', 5, 20, 1), 
                    'min_child_weight': hp.quniform('min_child_weight', 1, 2, 1),
                    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
                    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1),
                   }

In [ ]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from hyperopt import STATUS_OK

# XGBoost용 목적 함수 정의
# 핵심 포인트:
# 1) fmin()에서 입력된 search_space 값은 모두 실수형 → 정수형 하이퍼파라미터는 int() 캐스팅 필요
# 2) 정확도는 클수록 좋으므로 -1을 곱해 최소화 문제로 변환
def objective_func(search_space):
    # 수행 시간 절약을 위해 n_estimators는 100으로 축소
    xgb_clf = XGBClassifier(n_estimators=100, max_depth=int(search_space['max_depth']),
                            min_child_weight=int(search_space['min_child_weight']),
                            learning_rate=search_space['learning_rate'],
                            colsample_bytree=search_space['colsample_bytree'],
                            eval_metric='logloss')
    # 3-fold 교차검증으로 정확도 측정 → 평균 후 -1 곱해서 반환
    accuracy = cross_val_score(xgb_clf, X_train, y_train, scoring='accuracy', cv=3)
    
    return {'loss':-1 * np.mean(accuracy), 'status': STATUS_OK}

In [ ]:
from hyperopt import fmin, tpe, Trials

# 최대 50번 시도하며 최적 하이퍼파라미터 탐색
trial_val = Trials()
best = fmin(fn=objective_func,
            space=xgb_search_space,
            algo=tpe.suggest,
            max_evals=50, # 최대 반복 횟수 지정
            trials=trial_val, rstate=np.random.default_rng(seed=9))
print('best:', best)

In [ ]:
# 찾아낸 최적 하이퍼파라미터를 보기 좋게 출력
print('colsample_bytree:{0}, learning_rate:{1}, max_depth:{2}, min_child_weight:{3}'.format(
    round(best['colsample_bytree'], 5), round(best['learning_rate'], 5),
    int(best['max_depth']), int(best['min_child_weight'])))

In [ ]:
# 평가 함수 재정의 (셀 독립 실행 보장 + pred_proba 기반 AUC)
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score

def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
# HyperOpt가 찾아낸 최적 파라미터로 XGBoost 재학습
# 이번에는 n_estimators=400 + early stopping으로 충분히 학습
xgb_wrapper = XGBClassifier(n_estimators=400,
                            learning_rate=round(best['learning_rate'], 5),
                            max_depth=int(best['max_depth']),
                            min_child_weight=int(best['min_child_weight']),
                            colsample_bytree=round(best['colsample_bytree'], 5)
                           )

evals = [(X_tr, y_tr), (X_val, y_val)]
xgb_wrapper.fit(X_tr, y_tr, early_stopping_rounds=50, eval_metric='logloss',
                eval_set=evals, verbose=True)

preds = xgb_wrapper.predict(X_test)
pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

# 최종 성능 평가
get_clf_eval(y_test, preds, pred_proba)

---
## 정리 및 핵심 포인트

### 1) XGBoost
- GBM의 단점(속도, 과적합)을 보완한 부스팅 모델
- **파이썬 Native**(`xgb.train` + `DMatrix`)와 **사이킷런 Wrapper**(`XGBClassifier`) 두 가지 사용 방식 존재
- `early_stopping_rounds`로 과적합 방지 + 학습 시간 단축 가능 (단, 너무 작게 설정하면 underfitting 위험)

### 2) LightGBM
- XGBoost 대비 **빠른 학습 속도, 낮은 메모리 사용량**
- **Leaf-wise** 분할 방식 → 비대칭 깊은 트리 형성 → 적은 데이터에서는 과적합 위험
- 사이킷런 래퍼 `LGBMClassifier`를 사용하면 XGBoost와 거의 동일한 인터페이스로 사용 가능

### 3) HyperOpt (베이지안 최적화)
- 이전 시도 결과를 활용해 다음 탐색 지점을 추론 → Grid/Random Search보다 효율적
- 핵심 구성: **Search Space → Objective Function → fmin → Trials**
- 주의사항
  - `hp.quniform`은 실수 반환 → 정수형 파라미터는 **`int()` 변환 필수**
  - HyperOpt는 **최소화 문제만** 다루므로, 정확도/AUC처럼 최대화 지표는 **`-1` 곱해서** 반환
  - `cross_val_score`의 `scoring='accuracy'`는 양수 그대로, `neg_mean_squared_error` 같은 회귀 지표는 이미 음수로 반환됨

### 4) 세 가지를 함께 사용하는 흐름
1. XGBoost / LightGBM 등 모델 선택
2. HyperOpt로 **베이지안 최적화 기반 하이퍼파라미터 튜닝**
3. 찾은 최적 파라미터 + early stopping으로 **최종 모델 학습**
4. 테스트셋으로 평가 (`get_clf_eval` 등 활용)
